# demo_colony

> Phase 18 (`specs/multiagent/phase18/`): colony rules for large Claude teams with faulty members. Part 0's numpy forecast (committed first), the tracing check against Phase 15, the pilot, Stage 1 (N = 8 and 16), the go/no-go verdict, Stage 2 (N = 32 and 64, only on `go` and a raised cap) and cached reruns (`COLONY_PLAN`, `COLONY2_PLAN`).

Artifacts (gitignored): `results/multiagent/colony/{quick,pilot,stage1,stage2}/` (`runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md`, `colony.png`) and `results/multiagent/colony/forecast/` (Part 0). The response cache and spend ledger are Phase 10's (`results/multiagent/llm/`), so one API cap covers all live spend. Rows carry Phase 18's columns (`pooledBound`, `honestPooled`, the trace and goal drift) from `colony_trace.runColonyCell`.

- **Quick** (normal tests): Stage 1's 22 cells at one seed (N 8 and 16), 3 turns, with the fake client; identical at 1 and 8 processes.
- **Tracing check** (`--flags llm`, $0: Phase 14's cached replies): the outcome-based contamination against Phase 15's on Phase 14's N = 8 cells (AC-6).
- **Pilot**, **Stage 1**, **Stage 2** (`--cell-timeout 0 --flags llm`, with `EXPLRL_LLM_LIVE=1`): live, each behind its gate; `EXPLRL_COLONY_STAGE` (`pilot`, the default; `stage1`; `stage2`) sets how far a live run goes, so the pilot never runs straight into Stage 1.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np

from explRL.bench_grid import readCSV, writeCSV
from explRL.colony_grid import (COLONY2_PLAN, COLONY_PLAN, colonyPilotGrid, costRule1, costRule2, goNoGo, stage1Quick)
from explRL.colony_report import colonySummaries, figColony, writeColony
from explRL.colony_trace import runColonyCell
from explRL.coop_grid import coopHash
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.train import demo_artifact_dir

COLONY = 'multiagent/colony'
WORKERS = min(8, os.cpu_count() or 1)
LLM_WORKERS = 32
CAP = float(COLONY2_PLAN['cap'] or 150.0)            # the user raises it in COLONY2_PLAN before Stage 2
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
STAGE = os.environ.get('EXPLRL_COLONY_STAGE', 'pilot')    # how far a live run may go: 'pilot', 'stage1' or 'stage2'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
FORECAST = LLM.parents[2] / 'specs/multiagent/phase18/colony_predictions.json'     # Part 0 (gate for the pilot)


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def readLog(p):
    return [json.loads(x) for x in open(p)] if p.exists() else []


def runColony(mode, grids, spec, plan=COLONY_PLAN, workers=None, prior=()):
    """Cells through runWorkload (runColonyCell): runs.csv, turns.jsonl, timing.jsonl, report.md, colony.png, metrics.json;
    the analysis reads `prior` rows too (Stage 2: Stage 1's, for the tests and the trend over 8-64)."""
    live = spec.get('kind') == 'live'
    out = demo_artifact_dir(mode, COLONY)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload(grids, spec, workers or (LLM_WORKERS if live else WORKERS), str(logs), run=runColonyCell)
    wall = time.time() - t0
    rows = res['rows']
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    res['timing'] = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    S = colonySummaries(list(prior) + rows, plan, nboot=plan['nboot'] if live else 200)
    meta = dict(mode=mode, planHash=coopHash(plan), commit=gitCommit(), model=HAIKU, cap=CAP, wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)),
                calls=int(sum(r['calls'] for r in rows)), spent=Budget(CAP, spec['ledger']).spent() if spec.get('ledger') else 0.0)
    writeColony(out / 'report.md', S, meta)
    figColony(out, S)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'],
                                                      timing=res['timing'], summary=S), indent=2, sort_keys=True, default=str) + '\n')
    if live:
        for f in ('metrics.json', 'timing.jsonl'):
            if not (out / f.replace('.', '_live.')).exists():
                shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent in all, "
          f"{wall:.0f} s; stopped: {res['stopped']}")
    return rows, S, meta


def pilotCheck(rows):
    "Invalid rate, tokens and cost per call, and the plain board's honest loss (clean - liars) at the pilot seed."
    calls = sum(r['calls'] for r in rows) or 1
    v = {r['level']: r for r in rows}
    return dict(invalid=sum(r['invalid'] for r in rows) / calls, costPerCall=sum(r['cost'] for r in rows) / calls,
                inputPerCall=sum(r['inputTokens'] + r['cacheReadTokens'] + r['cacheWriteTokens'] for r in rows) / calls,
                outputPerCall=sum(r['outputTokens'] for r in rows) / calls,
                plainLoss=v['plain|clean']['welfareHonest'] - v['plain|independent']['welfareHonest'])

## Quick: Stage 1's cells with the fake client (pipeline check)

In [ ]:
t0 = time.time()
qcells = stage1Quick()
rows, S, _ = runColony('quick', [('colony1', qcells)], dict(kind='fake'))
assert time.time() - t0 < 300 and len(rows) == len(qcells) == 22
q = demo_artifact_dir('quick', COLONY)
for f in ('runs.csv', 'turns.jsonl', 'timing.jsonl', 'metrics.json', 'report.md', 'colony.png'):
    assert (q / f).stat().st_size > 0, f
read = lambda: tuple((q / f).read_bytes() for f in ('runs.csv', 'turns.jsonl'))
ref = read()
runColony('quick', [('colony1', qcells)], dict(kind='fake', pooled=False), workers=1)
assert read() == ref
assert all(np.isfinite(r['honestPooled']) and 'freeRiding' in r and 'contamination' in r for r in rows)
recs = readLog(q / 'turns.jsonl')
arm = lambda x: x['key'].split('|')[2]
assert all('(agent ' not in x['obs'] for x in recs if arm(x) in ('colony', 'colonyGuards'))       # the tally, not the plain list
assert {(r['level'], r['N'], r['nLiar']) for r in rows if r['level'] == 'colonyGuards|independent'} == {('colonyGuards|independent', 8, 2), ('colonyGuards|independent', 16, 4)}
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(q / 'report.md').read()[:1200])

## Part 0 (numpy): the committed forecast

`tools/colony_forecast.py run` (10,080 numpy cells); its predictions must be committed before the pilot.

In [ ]:
if FORECAST.exists():
    P = json.loads(FORECAST.read_text())
    for k, v in P['predictions'].items():
        print(f"{k}: {v['sign']} ({v['estimate']:.3f} [{v['ci'][0]:.3f}, {v['ci'][1]:.3f}])")
else:
    print('no forecast yet: python tools/colony_forecast.py run $PWD')

## Tracing check (`--flags llm`, $0): Phase 18's contamination against Phase 15's (AC-6)

Phase 14's baseline cells with liars at N 8, 16 and 32, replayed from the cache: the outcome-based contamination (`colony_trace`: answers built on a lie) against Phase 15's exposure-based one (`liar_report.mechanics`). AC-6 (revised after measuring): never above Phase 15's in any cell, and rising with N as Phase 15's does; the gap is exposure without uptake.

In [ ]:
#| llm
from explRL.liar_report import mechanics
from explRL.liar_trace import cacheOnlyClient, traceRun
from explRL.colony_trace import replay, traceBoard
from explRL.llm_runs import LLMConfig
from explRL.scale_grid import scaleGrid

if not (LLM / 'cache').exists():
    print('skipped: no response cache')
else:
    means = {}
    for N in (8, 16, 32):
        pairs15 = []
        for c in [c for c in scaleGrid() if c['N'] == N and c['level'] == 'baseline|liar']:
            cfg = LLMConfig.fromDict(c['cfg'])
            tr = traceRun(cfg, c['seed'], cacheOnlyClient(cfg, LLM / 'cache'))
            rp = replay(cfg, c['seed'], cacheOnlyClient(cfg, LLM / 'cache'))
            assert rp.rows == tr.rows                                   # the replay is the run
            pairs15.append((mechanics(tr.trace, tr.board, tr.nLiar)['contaminated'], traceBoard(rp.board, rp.answers, rp.nLiar)['contamination']))
        a, b = np.array(pairs15).T
        assert (b <= a + 1e-12).all(), N
        means[N] = (a.mean(), b.mean())
        print(f'N {N}: Phase 15 {a.mean():.3f}, Phase 18 {b.mean():.3f}, gap per cell {np.round(a - b, 3)}')
    assert means[32][1] > means[8][1] and means[32][0] > means[8][0]   # the same trend

## Pilot (`--flags llm`, live, about 2 200 calls, about $1)

N = 16, `plain` and `colony`, clean and with independent liars, seed 1890 (not analysed). Gates: the forecast committed and `COLONY_PLAN` frozen. AC-8: invalid replies under 5%, cost per call, the plain board's honest loss (stop and ask if none), the cost rule.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert FORECAST.exists(), 'commit the Part 0 forecast before the pilot'
    assert COLONY_PLAN['frozen'], 'freeze and commit COLONY_PLAN before the pilot'
    prows, _, _ = runColony('pilot', [('colonyPilot', colonyPilotGrid())], LIVE_SPEC)
    pc = pilotCheck(prows)
    rule = costRule1(pc['costPerCall'])
    print(f"invalid {pc['invalid']:.3f} (band < {COLONY_PLAN['pilot']['bands']['invalid']}); tokens per call {pc['inputPerCall']:.0f} in, "
          f"{pc['outputPerCall']:.0f} out; cost per call ${pc['costPerCall']:.5f}; plain board's honest loss {pc['plainLoss']:.3f}")
    print(f"projected Stage 1: ${rule['projected']:.2f}; cost rule: {rule['verdict']}")

## Stage 1 (`--cell-timeout 0 --flags llm`, live, 176 cells, about $25)

The cells the pilot's cost rule allows; then the summaries, the go/no-go verdict, and the forecast's score (`tools/colony_forecast.py score`).

In [ ]:
#| llm
def stage1Cells():
    "The cells the cost rule allows from the pilot (None: stop and ask)."
    pr = demo_artifact_dir('pilot', COLONY) / 'runs.csv'
    if not pr.exists():
        return None
    pc = pilotCheck(readCSV(pr))
    if not pc['invalid'] < COLONY_PLAN['pilot']['bands']['invalid'] or not pc['plainLoss'] > 0:
        print(f"pilot: invalid {pc['invalid']:.3f}, plain board's honest loss {pc['plainLoss']:.3f}: stop and ask")
        return None
    rule = costRule1(pc['costPerCall'])
    print(f"cost rule: {rule['verdict']} (projected ${rule['projected']:.2f})")
    return rule['cells']


if not LIVE or STAGE not in ('stage1', 'stage2'):
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage1 (after the pilot\'s verdict is recorded)')
else:
    assert COLONY_PLAN['frozen'], 'COLONY_PLAN must be frozen'
    cells = stage1Cells()
    if cells is None:
        print('Stage 1 not run: stop and ask')
    else:
        rows1, S1, _ = runColony('stage1', [('colony1', cells)], LIVE_SPEC)
        print(open(demo_artifact_dir('stage1', COLONY) / 'report.md').read()[:4000])
        print('go/no-go:', goNoGo(rows1))

## Stage 2 (`--cell-timeout 0 --flags llm`, live, about $77-108; only on `go` and a raised cap)

In [ ]:
#| llm
if not LIVE or STAGE != 'stage2':
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage2')
elif not (COLONY2_PLAN['frozen'] and COLONY2_PLAN['capRaised'] and COLONY2_PLAN['arm']):
    print('skipped: Stage 2 needs a go verdict, the user-raised cap and COLONY2_PLAN frozen with the chosen arm')
else:
    pr = readCSV(demo_artifact_dir('pilot', COLONY) / 'runs.csv')
    rule = costRule2(pilotCheck(pr)['costPerCall'] * COLONY2_PLAN['priceRatio'], COLONY2_PLAN['arm'])
    print(f"Stage 2 cost rule: {rule['verdict']} (projected ${rule['projected']:.2f})")
    rows1 = readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv')
    plan2 = dict(COLONY_PLAN, sizes=COLONY_PLAN['sizes'] + COLONY2_PLAN['sizes'], stage2=dict(arm=COLONY2_PLAN['arm'], sizes=COLONY2_PLAN['sizes']))
    rows2, S2, _ = runColony('stage2', [('colony2', rule['cells'])], LIVE_SPEC, plan=plan2, prior=rows1)
    print('trend (H18-5, confirmatory):', S2.get('trend'))
    print(open(demo_artifact_dir('stage2', COLONY) / 'report.md').read()[:4000])

## Cached reruns (`--flags llm`, after the live runs)

Every reply from the cache: `runs.csv` and `turns.jsonl` byte-identical, nothing spent (AC-10).

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1')
else:
    def stage2Cells():
        if not (COLONY2_PLAN['frozen'] and COLONY2_PLAN['arm']):
            return None
        pr = readCSV(demo_artifact_dir('pilot', COLONY) / 'runs.csv')
        return costRule2(pilotCheck(pr)['costPerCall'] * COLONY2_PLAN['priceRatio'], COLONY2_PLAN['arm'])['cells']

    for mode, grid in (('pilot', lambda: colonyPilotGrid()), ('stage1', stage1Cells), ('stage2', stage2Cells)):
        files = [demo_artifact_dir(mode, COLONY) / f for f in ('runs.csv', 'turns.jsonl')]
        if not files[0].exists() or (cells := grid()) is None:
            print(f'{mode}: skipped, no live run')
            continue
        before, spent = [p.read_bytes() for p in files], Budget(CAP, LIVE_SPEC['ledger']).spent()
        study = dict(pilot='colonyPilot', stage1='colony1', stage2='colony2')[mode]
        plan = COLONY_PLAN if mode != 'stage2' else dict(COLONY_PLAN, sizes=COLONY_PLAN['sizes'] + COLONY2_PLAN['sizes'],
                                                          stage2=dict(arm=COLONY2_PLAN['arm'], sizes=COLONY2_PLAN['sizes']))
        prior = readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv') if mode == 'stage2' else ()
        runColony(mode, [(study, cells)], LIVE_SPEC, plan=plan, prior=prior)
        assert [p.read_bytes() for p in files] == before and Budget(CAP, LIVE_SPEC['ledger']).spent() == spent, mode
        print(f'{mode}: cached rerun byte-identical, free')